In [1]:
"""
Лабораторная работа № 4.2.
Работу выполнила: Михалева Полина Вячеславовна, ЦИБ-251
Вариант 17: Автосалон
Дочерние классы 1 (Наследуют БК 1): Car (тип кузова), Motorcycle (объем двиг.)
Дочерний класс 2 (Наследует БК 2): ReturningBuyer (история покупок)
Контейнер: Sale
Полиморфный метод (Бизнес-логика): calculate_trade_in_value() (разный % от цены)

"""
from typing import List

# --- БАЗОВЫЕ КЛАССЫ (с валидацией из 4.1) ---
class Vehicle:
    def __init__(self, model: str, price: float):
        if price < 0: raise ValueError("Цена не может быть отрицательной.")
        self.model = model
        self.price = price
    def __str__(self) -> str:
        return f"{self.model} (Цена: {self.price:,.2f} руб.)"

class Buyer:
    def __init__(self, buyer_id: int, name: str):
        if buyer_id <= 0: raise ValueError("ID должен быть > 0.")
        self.buyer_id = buyer_id
        self.name = name
    def __str__(self) -> str:
        return f"{self.name} (ID: {self.buyer_id})"


# --- ДОЧЕРНИЕ КЛАССЫ (Наследование и Полиморфизм) ---
class Car(Vehicle):
    def __init__(self, model: str, price: float, body_type: str):
        super().__init__(model, price)
        self.body_type = body_type

    def calculate_trade_in_value(self) -> float:
        return self.price * 0.75

    def __str__(self) -> str:
        return f"Авто: {self.model} ({self.body_type}) | Trade-in: {self.calculate_trade_in_value():,.2f} руб."


class Motorcycle(Vehicle):
    def __init__(self, model: str, price: float, engine_volume: float):
        super().__init__(model, price)
        self.engine_volume = engine_volume

    def calculate_trade_in_value(self) -> float:
        return self.price * 0.60

    def __str__(self) -> str:
        return f"Мото: {self.model} ({self.engine_volume}л) | Trade-in: {self.calculate_trade_in_value():,.2f} руб."


class ReturningBuyer(Buyer):
    def __init__(self, buyer_id: int, name: str, previous_purchases: int):
        super().__init__(buyer_id, name)
        self.previous_purchases = previous_purchases

    def __str__(self) -> str:
        return f"Постоянный клиент: {self.name} (Покупок: {self.previous_purchases})"


# --- КЛАСС-КОНТЕЙНЕР (Композиция + Защита типов) ---
class Sale:
    def __init__(self, buyer: Buyer):
        if not isinstance(buyer, Buyer):
            raise TypeError("Покупатель должен быть объектом класса Buyer или его наследника.")
        self.buyer = buyer
        self.vehicles: List[Vehicle] = []

    def add_vehicle(self, vehicle: Vehicle) -> None:
        # --- ЗАЩИТА ТИПОВ (Type Checking) ---
        if not isinstance(vehicle, Vehicle):
            raise TypeError("В сделку можно добавить только объект класса Vehicle или его наследника (Car/Motorcycle).")
        self.vehicles.append(vehicle)

    def calculate_total_trade_in(self) -> float:
        total_value = 0.0
        for vehicle in self.vehicles:
            total_value += vehicle.calculate_trade_in_value() # Полиморфный вызов

        if isinstance(self.buyer, ReturningBuyer):
            total_value += total_value * 0.05 # Бонус 5%

        return total_value

    def generate_report(self) -> None:
        print("\n" + "="*50)
        print(f"ОТЧЕТ ПО СДЕЛКЕ для: {self.buyer}")
        print("-" * 50)
        for v in self.vehicles:
            print(f"  • {v}")
        print("-" * 50)
        print(f"ИТОГО: {self.calculate_total_trade_in():,.2f} руб.")
        print("="*50 + "\n")


# --- ДЕМОСТРАЦИЯ СЦЕНАРИЯ ---
if __name__ == "__main__":
    print("=== 1. Корректный бизнес-процесс ===")
    buyer1 = ReturningBuyer(buyer_id=102, name="Мария", previous_purchases=3)
    car1 = Car(model="BMW X5", price=8000000.0, body_type="Внедорожник")
    moto1 = Motorcycle(model="Harley", price=1200000.0, engine_volume=0.883)

    sale1 = Sale(buyer=buyer1)
    sale1.add_vehicle(car1)
    sale1.add_vehicle(moto1)
    sale1.generate_report()

    print("=== 2. Демонстрация обработки ошибок (try...except) ===")

    # Сценарий А: Попытка добавить в сделку не тот тип данных (например, строку вместо машины)
    try:
        print("Пытаемся добавить в сделку обычную строку вместо автомобиля...")
        sale1.add_vehicle("Это просто текст, а не объект Vehicle")
    except TypeError as e:
        print(f"Перехвачена ошибка TypeError: {e}")

    # Сценарий Б: Попытка создать сделку без покупателя или с неправильным объектом
    try:
        print("Пытаемся создать сделку, передав вместо покупателя число...")
        bad_sale = Sale(buyer=999)
    except TypeError as e:
        print(f"Перехвачена ошибка TypeError: {e}")


=== 1. Корректный бизнес-процесс ===

ОТЧЕТ ПО СДЕЛКЕ для: Постоянный клиент: Мария (Покупок: 3)
--------------------------------------------------
  • Авто: BMW X5 (Внедорожник) | Trade-in: 6,000,000.00 руб.
  • Мото: Harley (0.883л) | Trade-in: 720,000.00 руб.
--------------------------------------------------
ИТОГО: 7,056,000.00 руб.

=== 2. Демонстрация обработки ошибок (try...except) ===
Пытаемся добавить в сделку обычную строку вместо автомобиля...
Перехвачена ошибка TypeError: В сделку можно добавить только объект класса Vehicle или его наследника (Car/Motorcycle).
Пытаемся создать сделку, передав вместо покупателя число...
Перехвачена ошибка TypeError: Покупатель должен быть объектом класса Buyer или его наследника.
